# Analys av ljudnivådata — Steg 2

Läser de Parquet-filer som `01_ingest_USBdata.ipynb` sparade (timstatistik,
kuvert och peaks) och gör all analys och plotting på dem. Hela årets data tar
då bara några MB i minnet.

Behov av full upplösning för ett avgränsat intervall (t.ex. medelvärde över
ett valt tidsfönster eller detaljerad plot av ett dygn) hanteras genom att
läsa in aktuell/e dagsfil(er) på begäran — se avsnittet längst ner.


In [ ]:
%matplotlib ipympl
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import os
from math import log10

from se3_common import (load_calibration, find_files, file_date,
                        read_day, adjust_day)

In [ ]:
# Sökvägar och parametrar
datapath = r'C:\1_Validering\Buller\BG_2025\USBdata'   # för inläsning i full upplösning
outpath  = r'C:\1_Validering\Buller\BG_2025\agg'
exppath  = r'C:\1_Validering\Buller\BG_2025'            # slutexport av CSV

location = 'Bergsgatan 17'

# Korrigeringarna (kalibrering + frifält) är redan tillämpade i steg 1
free_field_comp      = 0
min_samples_per_hour = 8*60*45   # 45 min vid 8 Hz (8 sampel/s); timmar med färre sampel används inte för LAeq


In [ ]:
# Läs in den aggregerade datan
hourly      = pd.read_parquet(os.path.join(outpath, 'hourly_stats.parquet'))
envelope_df = pd.read_parquet(os.path.join(outpath, 'envelope.parquet'))
peaks_df    = pd.read_parquet(os.path.join(outpath, 'peaks.parquet'))

# Timvärden i decibel (exakt energimedelvärde från summan av 10^(L/10))
hourly['LAeq'] = np.where(
    hourly['count'] > min_samples_per_hour,
    10 * np.log10(hourly['sum_log'] / hourly['count']) - free_field_comp,
    np.nan,
)
hourly['LAFmax'] = 10 * np.log10(hourly['max_log']) - free_field_comp

hourly.info()


## Tim-, dygns- och veckovisa nivåer


In [ ]:
# Medelvärden per timme
hourly.plot(y=['LAeq', 'LAFmax'], figsize=(9.8, 5), linewidth=0.5, grid=True)


In [ ]:
# Medelvärden per dygn (exakt, beräknat från summorna)
daily = pd.DataFrame({
    'LAeq':   10 * np.log10(
        hourly['sum_log'].resample('D').sum() / hourly['count'].resample('D').sum()
    ) - free_field_comp,
    'LAFmax': hourly['LAFmax'].resample('D').max(),
})
daily.plot(y=['LAeq', 'LAFmax'], figsize=(9.8, 5), linewidth=0.5, grid=True)


In [ ]:
# Medelvärden per vecka
weekly = pd.DataFrame({
    'LAeq':   10 * np.log10(
        hourly['sum_log'].resample('W').sum() / hourly['count'].resample('W').sum()
    ) - free_field_comp,
    'LAFmax': hourly['LAFmax'].resample('W').max(),
})
weekly.plot(y=['LAeq', 'LAFmax'], figsize=(9.8, 5), linewidth=0.5, grid=True)


## Översikt och peaks


In [ ]:
# Översikt: kuvert (min/medel/max per 10 min) med peaks markerade
fig, ax = plt.subplots(figsize=(9.8, 5))
ax.fill_between(envelope_df.index, envelope_df['min'], envelope_df['max'],
                alpha=0.3, linewidth=0, label='min–max')
ax.plot(envelope_df.index, envelope_df['mean'], linewidth=0.5, label='medel')
ax.plot(peaks_df['Peak_Height'], 'o', markersize=3, color='tab:red',
        label='passager >80 dB(A)')
ax.set_ylabel('dB(A)')
ax.set_title(f'Ljudnivå (LAF) vid {location}')
ax.legend()
ax.grid()


In [ ]:
# Antal bullrande passager per timme
peaks_df.resample('h').count().plot(
    y='Peak_Height',
    legend=None,
    figsize=(9.8, 5),
    linewidth=0.5,
    grid=True,
    ylabel="Antal bullrande passager >80 dB(A)",
    title=f"Antal bullrande passager per timme vid {location}",
)


In [ ]:
# Antal bullrande passager per dygn i valt tidsfönster
def passager_per_dygn(t1, t2):
    plot_df = peaks_df.between_time(t1, t2).resample('D').count()
    if plot_df.empty:
        print(f"Inga passager kl {t1[:5]}-{t2[:5]}")
        return
    plot_df.plot(
        y='Peak_Height',
        legend=None,
        figsize=(9.8, 5),
        grid=True,
        ylabel="Antal bullrande passager >80 dB(A)",
        title=f"Antal bullrande passager kl {t1[:5]}-{t2[:5]} vid {location}",
    )


passager_per_dygn('23:00', '00:00')
passager_per_dygn('22:00', '23:00')
passager_per_dygn('23:00', '05:00')


In [ ]:
# Antal bullrande passager per dygn
peaks_df.resample('D').count().plot(
    y='Peak_Height',
    legend=None,
    figsize=(9.8, 5),
    grid=True,
    ylabel="Antal bullrande passager >80 dB(A)",
    title=f"Antal bullrande passager per dygn vid {location}",
)


In [ ]:
# Antal bullrande passager per vecka
peaks_df.resample('W').count().plot(
    y='Peak_Height',
    legend=None,
    figsize=(9.8, 5),
    ylim=[0, 650],
    grid=True,
    ylabel="Antal bullrande passager >80 dB(A)",
    title=f"Antal bullrande passager per vecka vid {location}",
)


## Export till CSV


In [ ]:
# Antal passager per dygn
peaks_df.resample('D').count().to_csv(
    sep=';', decimal=',', date_format='%Y-%m-%d %H:%M',
    path_or_buf=os.path.join(exppath, 'BG_per_dygn.csv'),
)

# Timvärden
hourly[['LAeq', 'LAFmax', 'count']].to_csv(
    sep=';', decimal=',', date_format='%Y-%m-%d %H:%M',
    path_or_buf=os.path.join(exppath, 'BG_timmar.csv'),
)

# Peakedata
peaks_df.to_csv(
    sep=';', decimal=',', date_format='%Y-%m-%d %H:%M',
    path_or_buf=os.path.join(exppath, 'Peaks_BG.csv'),
)


## Full upplösning på begäran

För ett avgränsat intervall läses endast berörda dagsfiler in. Minnet räcker
då gott även om hela året är bearbetat, eftersom en eller ett fåtal dygn
(~50 MB per dygn) läses in åt gången.


In [ ]:
# Kalibrering och sommartidsjustering från steg 1 läses från aggmappen,
# så fullupplösningsdata justeras identiskt med den aggregerade datan
dst_shifts = load_calibration(os.path.join(outpath, 'calibration.json'))
free_field_comp_full = 6


def load_fullres(start, end):
    """Läs in rådata med full upplösning för intervallet start–end."""
    start, end = pd.Timestamp(start), pd.Timestamp(end)
    frames = []
    for fname in find_files(datapath):
        fdate = file_date(fname)
        if fdate <= end and fdate + pd.Timedelta(days=1) >= start:
            frames.append(read_day(datapath, fname))
    df = pd.concat(frames).sort_index()
    return adjust_day(df, free_field_comp_full, dst_shifts).loc[start:end]

In [ ]:
# Exempel: medelvärde över valt tidsfönster
start = pd.Timestamp("2026-03-27 00:00:00")
end   = pd.Timestamp("2026-03-27 06:00:00")

df_subset = load_fullres(start, end)
log_lin = 10 ** (df_subset['LAF'] / 10)

# Kalibrering och frifältskompensering är redan tillämpade i load_fullres
dfavg = pd.Series({
    'LAeq':   10 * log10(log_lin.mean()),
    'LAFmax': 10 * log10(log_lin.max()),
    'count':  log_lin.count(),
})
dfavg


In [ ]:
# Exempel: full upplösningsplot för ett dygn, med peaks markerade
df_day = load_fullres('2026-03-27', '2026-03-28')

fig, ax = plt.subplots(figsize=(9.8, 5))
ax.plot(df_day['LAF'], linewidth=0.5)
day_peaks = peaks_df.loc['2026-03-27':'2026-03-28', 'Peak_Height']
ax.plot(day_peaks, 'o', color='tab:red')
ax.set_ylabel('dB(A)')
ax.set_title(f"Ljudnivå (LAF) vid {location}, 2026-03-27")
ax.grid()
